# OPERA RTC-S1 Time-Series — Oosterweel Construction Works, Antwerp

This notebook demonstrates the full workflow for **OPERA Radiometric
Terrain-Corrected (RTC)** SAR backscatter analysis from Sentinel-1 over the
**Oosterweel Link construction site** in Antwerp, Belgium.

## Workflow

1. **Define AOI & working directory** — bounding box + where data lives on disk
2. **Choose archive** — Terrascope, NASA/ASF, or automatic fallback
3. **Search** — query the STAC catalogue for available passes
4. **Coverage analysis** — inspect spatial coverage per pass, filter by threshold
5. **Load & mosaic → disk** — stream pass-by-pass, save each to GeoTIFF
6. **Visualise from disk** — load one pass at a time for composites, GIFs, time-series

All mosaicked passes are saved as GeoTIFF files under `WORKDIR/passes/`.
Once saved, subsequent visualisation steps load data **one pass at a time**
from disk, keeping peak memory low regardless of the number of passes.

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from rs_tools.config import BoundingBox, SearchConfig
from rs_tools.search import search_archive
from rs_tools.datasets.catalog import get as get_dataset
from rs_tools.datasets.coverage import (
    summarize_search_results,
    print_coverage_report,
    filter_by_coverage,
    records_to_items,
)
from rs_tools.datasets.loader import (
    load_items,
    load_passes_from_disk,
    setup_terrascope_auth,
    LoadedItem,
)
from rs_tools.visualization.rtc_composite import rtc_composite
from rs_tools.visualization.scalebar import add_scalebar
from rs_tools.visualization.animation import save_timeseries_gif_lazy
from rs_tools.visualization.overlays import fetch_roads, overlay_roads, annotate_location, overlay_geojson

# Path to the Oosterweel construction trajectory GeoJSON (same directory as this notebook)
GEOJSON_PATH = "oosterweel_trajectory.geojson"

## 1. Working directory, AOI & temporal range

**`WORKDIR`** is where all data and outputs are stored:

```
WORKDIR/
  passes/                ← mosaicked GeoTIFF files (one folder per pass)
    20220116_0559_DES_T110/
      VV.tif
      VH.tif
      metadata.json
    20220120_1741_ASC_T037/
      ...
  gifs/                  ← animated GIF exports
  plots/                 ← static figures
```

Passes are named **chronologically** (`YYYYMMDD_HHMM_ORB_TRRR`) so they
sort naturally in the filesystem.

If passes already exist on disk from a previous run, the loader will
**skip** them automatically and only download missing or corrupt ones.

The Oosterweel Link is a major infrastructure project involving tunnels under the
Scheldt river connecting the left and right banks of Antwerp.

In [ ]:
# ── Working directory (all data & outputs stored here) ────────────────────
WORKDIR = os.path.join(os.getcwd(), "output", "oosterweel")
os.makedirs(os.path.join(WORKDIR, "passes"), exist_ok=True)
os.makedirs(os.path.join(WORKDIR, "gifs"), exist_ok=True)
os.makedirs(os.path.join(WORKDIR, "plots"), exist_ok=True)
print(f"Working directory: {WORKDIR}")

# ── Area of interest ─────────────────────────────────────────────────────
bbox_oosterweel = BoundingBox(west=4.30, south=51.17, east=4.48, north=51.27)

# Full Sentinel-1 mission period (OPERA data available from ~Oct 2021)
START_DATE = "2014-10-01"
END_DATE   = "2026-03-24"

print(f"AOI:        {bbox_oosterweel}")
print(f"Time range: {START_DATE} → {END_DATE}")

## 2. AOI overview map

In [ ]:
import cartopy.crs as ccrs
import matplotlib.patches as mpatches
from cartopy.io.img_tiles import OSM

osm_tiles = OSM()
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": osm_tiles.crs})
ax.set_extent([bbox_oosterweel.west - 0.05, bbox_oosterweel.east + 0.05,
               bbox_oosterweel.south - 0.02, bbox_oosterweel.north + 0.02],
              crs=ccrs.PlateCarree())
ax.add_image(osm_tiles, 12)
rect = mpatches.Rectangle(
    (bbox_oosterweel.west, bbox_oosterweel.south),
    bbox_oosterweel.east - bbox_oosterweel.west,
    bbox_oosterweel.north - bbox_oosterweel.south,
    transform=ccrs.PlateCarree(), linewidth=2,
    edgecolor="red", facecolor="red", alpha=0.15,
)
ax.add_patch(rect)
ax.set_title("Oosterweel construction works — AOI", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Choose archive

OPERA RTC-S1 data is available from multiple archives. Set `ARCHIVE` below
to control which source is queried:

| `ARCHIVE` value | Description |
|:---:|:---|
| `"terrascope"` | **Terrascope STAC** (VITO). Fast from Europe, uses local mounts on VITO servers. |
| `"nasa"` | **NASA ASF DAAC**. Original source; uses S3 on AWS, HTTPS elsewhere. |
| `None` | **Automatic fallback** — tries archives in catalog priority order (Terrascope → NASA) and uses the first one that returns results. |

On a **VITO server**, Terrascope data can be streamed from the local `/data/MTDA`
mount (fastest) or via HTTPS (if the local mount is unavailable on that node).
The library detects this automatically.

In [ ]:
# ── Archive selection ─────────────────────────────────────────────────────
#   "terrascope"  — Terrascope STAC only
#   "nasa"        — NASA ASF DAAC only
#   None          — automatic fallback (tries Terrascope first, then NASA)
ARCHIVE = "terrascope"

# Show which collections this dataset maps to in each archive
ds_info = get_dataset("OPERA_RTC_S1")
print(f"Dataset:  {ds_info.name}")
print(f"Product:  {ds_info.description}")
print(f"Resolution: {ds_info.spatial_resolution}, repeat cycle: {ds_info.temporal_resolution}")
print(f"\nAvailable archives:")
for arch, colls in ds_info.archive_collections.items():
    marker = " ← selected" if arch == ARCHIVE else ""
    print(f"  {arch:12s} → {', '.join(colls)}{marker}")

## 4. Search the catalogue

Query the STAC catalogue for all available passes over the full time range.
This does **not** download any pixel data — only lightweight metadata.

In [ ]:
# Search for all available STAC items
collections = ds_info.archive_collections[ARCHIVE]
config = SearchConfig(
    start_date=START_DATE,
    end_date=END_DATE,
    bbox=bbox_oosterweel,
    collections=collections,
    limit=500,
)
items = search_archive(ARCHIVE, config)
print(f"Search returned {len(items)} STAC items (burst granules)")

## 5. Coverage analysis

Group the raw burst granules by satellite pass (same track + same date) and
compute the **spatial coverage** — what percentage of the AOI bounding box is
covered by the union of all burst footprints in that pass.

This lets you decide:
- **Which coverage threshold** to apply (e.g. keep only passes with ≥ 80% coverage)
- **Which orbit direction** to keep (ascending, descending, or both)
- **Which tracks** to focus on

In [ ]:
# Build a per-pass coverage summary (no pixel data loaded yet)
records = summarize_search_results(items, bbox_oosterweel)
print_coverage_report(records)

## 6. Filter by coverage threshold

Use the coverage report above to decide on a minimum coverage percentage.
Passes below this threshold will be dropped before any pixel data is
downloaded.

You can also filter by orbit direction, track number, date range, or platform.

In [ ]:
# ── Coverage threshold ────────────────────────────────────────────────────
MIN_COVERAGE_PCT = 80.0         # drop passes that cover < 80% of the AOI

# Optional additional filters (set to None to disable):
ORBIT_DIRECTION  = None         # "ascending", "descending", or None for both
TRACK            = None         # e.g. 110, or None for all tracks

selected = filter_by_coverage(
    records,
    min_coverage_pct=MIN_COVERAGE_PCT,
    orbit_direction=ORBIT_DIRECTION,
    track=TRACK,
)
print(f"Kept {len(selected)} of {len(records)} passes "
      f"(coverage ≥ {MIN_COVERAGE_PCT}%)")

# Convert back to raw STAC items for loading
selected_items = records_to_items(selected)
print(f"→ {len(selected_items)} burst granules to load")

## 7. Load, mosaic & save to disk — pass-by-pass streaming

Data is loaded **one satellite pass at a time**: bursts are read eagerly,
merged into a single mosaic, clipped to the AOI, **saved as GeoTIFF on
disk**, and then freed from memory before the next pass is loaded.

Each pass ends up in its own folder:
```
WORKDIR/passes/20220116_0559_DES_T110/
    VV.tif          ← 30 m GeoTIFF, clipped to AOI
    VH.tif
    metadata.json   ← platform, orbit, CRS, pixel size, …
```

**Incremental**: if passes are already on disk from a previous run, they
are skipped automatically.  Corrupt or incomplete passes are detected and
re-downloaded.

> If all passes are already saved, this cell finishes almost instantly.

In [ ]:
# Configure GDAL auth for the selected archive
if ARCHIVE == "terrascope":
    from rs_tools.archives.local import is_on_vito
    if is_on_vito():
        print("Running on VITO — will use local paths if available")
        try:
            setup_terrascope_auth()
        except RuntimeError:
            print("HTTPS credentials not configured — local file access will be used")
    else:
        setup_terrascope_auth()
elif ARCHIVE == "nasa":
    from rs_tools.archives.nasa import configure_gdal_nasa
    configure_gdal_nasa()

# Load data pass-by-pass, save each pass to disk, free memory
data = load_items(
    selected_items,
    assets=["VV", "VH"],
    bbox=bbox_oosterweel,
    mosaic=True,
    output_dir=WORKDIR,   # ← save each pass to WORKDIR/passes/
)
print(f"\n→ Saved {len(data)} mosaicked passes to {WORKDIR}/passes/")

## 8. Reload passes from disk

Load pass metadata from the GeoTIFF files saved in the previous step.
**No pixel data is loaded yet** — only the metadata (date, platform, orbit, CRS).
Call `item.load()` on individual passes to read pixels when needed.

In [ ]:
# Reload pass metadata from disk (pixel data stays on disk until needed)
data = load_passes_from_disk(WORKDIR)
print(f"\nPasses available on disk:")
for i, item in enumerate(data, 1):
    print(f"  [{i:3d}] {item.label}  →  {item.pass_dir}")

## 9. Inspect loaded passes

In [ ]:
print(f"{'#':>3}  {'Platform':<14}  {'Orbit':5}  {'Date':20}  {'CRS':12}  {'Pixel':>6}  {'On disk'}")
print("-" * 90)
for i, item in enumerate(data, 1):
    orb = (item.orbit_direction or "?")[:3].upper()
    has_data = "loaded" if item.data else "disk"
    print(f"{i:3d}  {item.platform:<14}  {orb:5}  "
          f"{item.datetime:%Y-%m-%d %H:%M UTC}  {item.crs or 'N/A':12}  "
          f"{item.pixel_size_m or 0:5.0f}m  {has_data}")

## 10. Acquisition timeline

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
dates = [item.datetime for item in data]
ax.scatter(dates, [0] * len(dates), marker="|", s=300,
           c="darkorange", linewidths=2, label=f"Oosterweel ({len(dates)})")
for item in data:
    lbl = item.platform.split("-")[1] if "-" in item.platform else item.platform
    ax.annotate(lbl, (item.datetime, 0), fontsize=6, ha="center", va="top",
                xytext=(0, -4), textcoords="offset points", color="darkorange")
ax.set_yticks([])
ax.set_title("OPERA RTC-S1 acquisition timeline — Oosterweel")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 11. RTC composite — latest scene

Load the most recent pass **from disk**, render a false-colour RTC
composite with road overlay, **construction trajectory**, and construction
site labels, then free the pixel data.

In [ ]:
if data:
    item = data[-1]
    item.load()  # read VV.tif / VH.tif into memory

    vv = item.data["VV"].values
    vh = item.data["VH"].values

    fig, ax = plt.subplots(figsize=(12, 10))
    rgb = rtc_composite(vv, vh)
    ax.imshow(rgb, origin="upper")
    ax.set_axis_off()
    ax.set_title(f"Oosterweel — {item.label}", fontsize=13)

    # Road overlay
    try:
        roads = fetch_roads(
            bbox=(bbox_oosterweel.south, bbox_oosterweel.west,
                  bbox_oosterweel.north, bbox_oosterweel.east),
            highway_types="motorway|trunk|primary|secondary",
        )
        overlay_roads(ax, roads, item.data["VV"],
                      color="#888888", linewidth=0.4, alpha=0.5)
    except Exception as e:
        print(f"  (road overlay skipped: {e})")

    # Construction trajectory overlay
    TRAJECTORY_CATEGORIES = [
        "Scheldetunnel", "Kanaaltunnels", "Bypass R1",
        "Motorway R1 (under construction)",
        "Oosterweelknooppunt (construction zone)",
        "Scheldetunnel construction site",
        "Junction construction site",
    ]
    TRAJECTORY_STYLES = {
        "Scheldetunnel":                     {"color": "lime",   "linewidth": 2.5, "linestyle": "--"},
        "Kanaaltunnels":                     {"color": "cyan",   "linewidth": 2.5, "linestyle": "--"},
        "Bypass R1":                         {"color": "magenta","linewidth": 2.0},
        "Motorway R1 (under construction)":  {"color": "red",    "linewidth": 1.8},
        "Oosterweelknooppunt (construction zone)": {"color": "orange", "linewidth": 1.5, "alpha": 0.5},
        "Scheldetunnel construction site":   {"color": "lime",   "linewidth": 1.2, "alpha": 0.4},
        "Junction construction site":        {"color": "orange", "linewidth": 1.2, "alpha": 0.4},
    }
    if os.path.exists(GEOJSON_PATH):
        overlay_geojson(
            ax, GEOJSON_PATH, item.data["VV"],
            category_styles=TRAJECTORY_STYLES,
            filter_categories=TRAJECTORY_CATEGORIES,
        )
        ax.legend(loc="upper right", fontsize=7, framealpha=0.7,
                  facecolor="black", edgecolor="gray", labelcolor="white")
    else:
        print(f"  ⚠ GeoJSON not found: {GEOJSON_PATH}")
        print(f"    Run: python fetch_oosterweel_geojson.py")

    # Key locations
    annotations = [
        (4.3925, 51.2340, "Oosterweel\ntunnel north"),
        (4.3885, 51.2210, "Scheldt\ncrossing"),
        (4.4035, 51.2140, "Linkeroever\ntunnel south"),
        (4.4220, 51.2230, "Port of\nAntwerp"),
        (4.3560, 51.2170, "Antwerp\ncity centre"),
        (4.4150, 51.2470, "R1 ring\nmotorway"),
    ]
    for lon, lat, label in annotations:
        try:
            annotate_location(ax, lon, lat, label, item.data["VV"],
                              color="cyan", fontsize=7, markersize=3)
        except Exception:
            pass

    ax.text(0.02, 0.02, item.label, transform=ax.transAxes, fontsize=10,
            color="white", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.7),
            verticalalignment="bottom")
    if item.pixel_size_m:
        add_scalebar(ax, item.pixel_size_m)
    plt.tight_layout()
    plt.show()

    item.unload()  # free pixel data from memory

## 12. Animated GIF export — one frame at a time

Each pass is loaded from disk, rendered as a PNG frame, and
immediately freed.  Only the small palette-mode PIL frames (~0.5 MB
each) are kept in memory while writing the GIF.

In [ ]:
def _rtc_from_disk(item):
    """Load a pass from disk, create RGB composite, unload, return frame."""
    item.load()
    vv = item.data["VV"].values
    vh = item.data["VH"].values
    rgb = rtc_composite(vv, vh)
    label = item.label
    item.unload()
    return rgb, label

if data:
    gif_path = os.path.join(WORKDIR, "gifs", "oosterweel_rtc.gif")
    save_timeseries_gif_lazy(
        data, gif_path,
        composite_fn=_rtc_from_disk,
        title="Oosterweel — OPERA RTC-S1",
        pixel_size_m=data[0].pixel_size_m, fps=2,
    )

## 13. Interactive time-series slider

Loads each pass from disk on demand when the slider position changes.

In [ ]:
from matplotlib.widgets import Slider

if data:
    # Pre-load first frame
    data[0].load()
    _first_rgb = rtc_composite(data[0].data["VV"].values,
                                data[0].data["VH"].values)
    data[0].unload()

    fig, ax = plt.subplots(figsize=(9, 9))
    plt.subplots_adjust(bottom=0.15)

    im = ax.imshow(_first_rgb, origin="upper")
    ax.set_axis_off()
    ax.set_title(f"Oosterweel — {data[0].label}", fontsize=12)
    if data[0].pixel_size_m:
        add_scalebar(ax, data[0].pixel_size_m)

    txt = ax.text(0.02, 0.02, data[0].label, transform=ax.transAxes,
                  fontsize=10, color="white", fontweight="bold",
                  bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.7),
                  verticalalignment="bottom")
    del _first_rgb

    ax_slider = fig.add_axes([0.15, 0.04, 0.70, 0.03])
    slider = Slider(ax_slider, "Date", 0, len(data) - 1, valinit=0, valstep=1)

    def _update(val):
        idx = int(slider.val)
        item = data[idx]
        item.load()
        rgb = rtc_composite(item.data["VV"].values, item.data["VH"].values)
        item.unload()
        im.set_data(rgb)
        ax.set_title(f"Oosterweel — {item.label}", fontsize=12)
        txt.set_text(item.label)
        fig.canvas.draw_idle()

    slider.on_changed(_update)
    plt.show()

## 14. Backscatter time-series (dB)

Mean VV and VH backscatter converted to decibels.  Each pass is loaded
from disk one at a time.

Set **`POINT`** to a `(lon, lat)` tuple to extract the time-series at a
specific location (nearest pixel).  Set to `None` to compute the spatial
mean over the entire AOI.

In [ ]:
# ── Set to (lon, lat) for a point time-series, or None for AOI mean ───
POINT = (4.3925, 51.2340)   # Oosterweel tunnel north entrance
# POINT = None              # → spatial mean over entire AOI

def backscatter_db(items, point=None, crs="EPSG:4326"):
    """Extract VV/VH backscatter (dB) per pass.

    Parameters
    ----------
    items : list[LoadedItem]
        Passes with data on disk.
    point : tuple[float, float] or None
        (lon, lat) for point extraction; None = AOI spatial mean.
    crs : str
        CRS of *point* coordinates (default WGS-84).
    """
    import pyproj
    dates, vv_db, vh_db, sensors = [], [], [], []
    for item in items:
        item.load()
        vv_arr = item.data["VV"]
        vh_arr = item.data["VH"]

        if point is not None:
            # Reproject point to the raster CRS, then select nearest pixel
            raster_crs = vv_arr.rio.crs
            if raster_crs and str(raster_crs) != crs:
                transformer = pyproj.Transformer.from_crs(
                    crs, str(raster_crs), always_xy=True,
                )
                px, py = transformer.transform(point[0], point[1])
            else:
                px, py = point
            vv_val = float(vv_arr.sel(x=px, y=py, method="nearest").values)
            vh_val = float(vh_arr.sel(x=px, y=py, method="nearest").values)
        else:
            vv = vv_arr.values
            vh = vh_arr.values
            vv_val = float(np.nanmean(vv[vv > 0]))
            vh_val = float(np.nanmean(vh[vh > 0]))

        item.unload()

        if vv_val > 0 and vh_val > 0:
            dates.append(item.datetime)
            vv_db.append(10 * np.log10(vv_val))
            vh_db.append(10 * np.log10(vh_val))
            sensors.append(
                item.platform.split("-")[1] if "-" in item.platform else item.platform
            )
    return dates, vv_db, vh_db, sensors

if data:
    fig, ax = plt.subplots(figsize=(14, 5))
    dates, vv, vh, sensors = backscatter_db(data, point=POINT)

    ax.plot(dates, vv, "o-", color="steelblue", ms=4, lw=1, label="VV (dB)")
    ax.plot(dates, vh, "s-", color="darkorange", ms=4, lw=1, label="VH (dB)")
    for d, v, s in zip(dates, vv, sensors):
        ax.annotate(s, (d, v), fontsize=6, ha="center", va="bottom",
                    xytext=(0, 3), textcoords="offset points", color="steelblue")
    ax.set_ylabel("Backscatter (dB)")
    ax.set_xlabel("Acquisition date")
    if POINT:
        ax.set_title(f"Oosterweel — VV / VH backscatter at ({POINT[0]:.4f}, {POINT[1]:.4f})")
    else:
        ax.set_title("Oosterweel — VV / VH backscatter (AOI mean)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()